# claudesub

> Spawn a headless Claude child on this session's compacted history

In [ ]:
#| default_exp claudesub

`claudesub` starts a headless `claude -p` process to work from the parent's conversation. It gives the child a compact document instead of the full transcript. The document uses llmsurgery's compaction format to retain decisions and evidence with fewer tokens.

The child runs in its own process with its own clikernel kernel. This differs from Claude Code subagents that share the parent's kernel in this setup. It also avoids giving a fork the parent's full context to read on each turn.

Launch the child through a background Bash call or a Monitor. You can read its progress and resume it with an answer after it reports a blocker. This module supports Claude Code. It combines `fastclaude.session`, `llmsurgery.ant.prepare_compaction`, and the dojo completion registry.

In [ ]:
#| export
import json, os, subprocess, sys
from fastcore.utils import *
from fastcore.script import call_parse
from fastclaude.session import *
from fastclaude.core import claude_env
from llmsurgery.ant import prepare_compaction
from llmdojo.dojo import _completions, dojo_version
from llmdojo.tmpl import launch_config

In [ ]:
from fastcore.test import *
import shutil, tempfile

## The parent session

`parent_sid` reads `CLAUDE_CODE_SESSION_ID` from the launching process's environment. In the tested interactive sessions, subagents, and headless children, this ID matches the transcript filename. The function raises if the variable is unset or the transcript doesn't exist. It never guesses which conversation to compact.

The child skips the practice round using a dojo completion ID. `dojo_cid` selects the newest receipt for the current tooling version. If none exists, play a clean round in the parent before building the child command.

In [ ]:
#| export
def parent_sid(
    cwd=None, # Project directory; the current directory if None
):
    "Return the launching session ID. Raise if it is unset or has no transcript."
    sid = os.environ.get('CLAUDE_CODE_SESSION_ID')
    if not sid: raise RuntimeError('CLAUDE_CODE_SESSION_ID is unset: claudesub runs from inside a Claude Code session')
    if not (sess_dir(cwd)/f'{sid}.jsonl').exists(): raise FileNotFoundError(f'No transcript for session {sid} under {sess_dir(cwd)}')
    return sid

def dojo_cid():
    "Return the newest completion ID for this dojo version, or None."
    v = dojo_version()
    ids = [(o['t'],k) for k,o in _completions().items() if o.get('v')==v]
    return max(ids)[1] if ids else None

In [ ]:
os.environ['CLAUDE_CODE_SESSION_ID'] = 'not-a-session'
test_fail(parent_sid, contains='not-a-session')
del os.environ['CLAUDE_CODE_SESSION_ID']
test_fail(parent_sid, contains='CLAUDE_CODE_SESSION_ID')

In [ ]:
cid = dojo_cid()
assert cid is None or len(cid)==4
cid

## The child session

`prepare_compaction` renders the parent's conversation as a compact document. It returns a compaction boundary, a summary, and three records for the visible `/compact` exchange. `prep_sub` saves these five records under a new session ID.

The child session stays in the parent's project. A headless resume looks for its transcript there. Claude loads the history after the compaction boundary. The child starts with the compact document and any context the harness adds at launch, without a copy of the full parent transcript.

We'll check the records using a synthetic parent:

In [ ]:
#| export
def prep_sub(
    sid=None, # Parent session id; `parent_sid()` if None
    cwd=None, # Project directory; the current directory if None
):
    "Write a session holding only the parent's compacted history, returning its id for the child to resume"
    c = prepare_compaction(sid or parent_sid(cwd), cwd or '.')
    return save_sess(list(c.records), cwd=cwd, ts=True)

In [ ]:
mproj = Path(tempfile.mkdtemp())
turns = tool_turn('Which module handles retries?', 'mcp__clikernel__py', dict(code="rg('retry', 'src')"),
    'src/api.py:12:def with_retry(', 'Retries live in src/api.py.', cwd=mproj)
psid = save_sess(turns, cwd=mproj)
csid = prep_sub(psid, mproj)
child = load_sess(csid, mproj)
test_ne(csid, psid)
test_eq(child[0].subtype, 'compact_boundary')
assert child[1].isCompactSummary and 'retry' in rec_txt(child[1])
test_eq({r.sessionId for r in child}, {csid})
[r.type for r in child]

['system', 'user', 'user', 'user', 'user']

## The protocol and the command

`sub_cmd` passes the child's standing instructions through `--append-system-prompt`, separate from its task directive. These instructions don't become a conversation turn. Every child uses the same protocol with its chosen dojo completion ID.

The protocol explains the child's private kernel and receipt-based bootstrap. If blocked, the child must stop and report what it needs rather than wait for an answer. The parent can then resume it. Resuming restarts the child's kernel, though its conversation remains. The child must finish independent work before stopping and leave a report the parent can use without reading the transcript.

Standing arguments come from `launch_config('claude')`. The caller's extra flags follow them.

In [ ]:
#| export
PROTOCOL = """You are a subagent spawned by another Claude Code session with `claudesub`. The history above is that session's compacted transcript: you hold its decisions and evidence, but none of its kernel state. Your clikernel kernel is your own; no other session shares it. No person is watching, so never wait for an answer or ask a question mid-task: if you are blocked, stop and report exactly what you need, and the parent can resume you with `claudesub -r <your session id> '<answer>'`, which restarts your kernel, so finish whatever does not depend on the answer before you stop. Bootstrap with dojo_start({cid!r}). Then do the directive, and end with a report the parent can act on without reading your transcript: what you did, what you found, and anything left undone."""

def sub_cmd(
    directive, # The child's task
    sid, # Session id for the child to resume
    cid=None, # Dojo completion id to hand the child; `dojo_cid()` if None
    extra=(), # Further `claude` arguments, after the standing ones from `launch_config`
):
    "Build a headless resume command with streaming JSON output. Require a completion ID."
    cid = cid or dojo_cid()
    if not cid: raise RuntimeError('No clean dojo round is on record for this tooling version: play one in the parent session first')
    return ['claude', '-p', directive, f'--resume={sid}', '--output-format=stream-json', '--verbose',
        '--append-system-prompt', PROTOCOL.format(cid=cid), *launch_config('claude'), *extra]

In [ ]:
argv = sub_cmd('Summarize src/api.py', csid, cid='ab12', extra=['--model=haiku'])
test_eq(argv[:3], ['claude', '-p', 'Summarize src/api.py'])
assert f'--resume={csid}' in argv and argv[-1]=='--model=haiku'
assert "dojo_start('ab12')" in argv[argv.index('--append-system-prompt')+1]
os.environ['LLMDOJO_STATE_DIR'] = tempfile.mkdtemp()
test_fail(lambda: sub_cmd('x', csid), contains='play one in the parent session')
del os.environ['LLMDOJO_STATE_DIR']
argv[3:6]

['--resume=683e782a-7b9e-4824-a03d-29bd569be4b1',
 '--output-format=stream-json',
 '--verbose']

## Running the child

`run_sub` prints the child's visible text as it arrives. With a background Bash call, this output goes to the task's file. A Monitor can show each line as an event. `quiet=True` suppresses progress text and prints the final report instead.

The footer identifies the child session, turn count, and cost. The function returns the result event. It raises if the child exits without a result.

In [ ]:
#| export
def run_sub(
    argv, # The child command, from `sub_cmd`
    quiet=False, # Print only the final report and footer, not the child's text as it arrives?
    cwd=None, # Directory the child runs in; the current directory if None
):
    "Run the child, streaming its text, and return its result event"
    p = subprocess.Popen(argv, stdout=subprocess.PIPE, text=True, env=claude_env(), cwd=cwd)
    res = None
    for line in p.stdout:
        try: e = json.loads(line)
        except json.JSONDecodeError: continue
        if e.get('type')=='result': res = e
        elif e.get('type')=='assistant' and not quiet:
            for b in e['message'].get('content', []):
                if b.get('type')=='text' and b['text'].strip(): print(b['text'].strip(), flush=True)
    p.wait()
    if res is None: raise RuntimeError(f'claude exited {p.returncode} without a result')
    if quiet: print(res.get('result', ''))
    print(f"[claudesub {res['subtype']}] session {res['session_id']}, {res['num_turns']} turns, ${res.get('total_cost_usd', 0):.2f}", flush=True)
    return res

This acceptance test asks about a fact from the synthetic parent's history. It runs a real model and costs money. Its `eval: false` flag excludes it from normal notebook execution.

In [ ]:
#| eval: false
res = run_sub(sub_cmd('From the history only: which file holds the retries? Reply in one line.', csid, extra=['--model=haiku']), cwd=mproj)
assert 'api.py' in res['result']

## The command line

Run `claudesub 'directive'` to start a child. Use `claudesub -r <child-sid> 'answer'` to continue one that stopped for help. The resumed child keeps its conversation context.

The CLI uses `call_parse(nested=True)` to forward unknown arguments to `claude`. Write them as `--flag=value`, such as `--model=haiku` or `--max-turns=20`. A failed result makes the CLI exit nonzero. Background task notifications can report that failure.

In [ ]:
#| export
@call_parse(nested=True, pos=['directive'])
def main(
    directive:str, # The child's task
    Resume:str=None, # A child session id to continue with `directive` as its next prompt, instead of spawning from this session
    sid:bool=False, # Print the prepared child session id instead of launching
    Quiet:bool=False, # Print only the child's final report, not its progress text
    cwd:str=None, # Project directory to work from, for the parent lookup, the child session, and the child itself; the current directory if None
):
    "Start or resume a headless Claude child and print its progress and result."
    if cwd: os.chdir(cwd)
    s = Resume or prep_sub()
    if sid: return print(s)
    res = run_sub(sub_cmd(directive, s, extra=sys.argv[1:]), quiet=Quiet)
    if res['subtype'] != 'success': sys.exit(1)

## Cleanup

In [ ]:
shutil.rmtree(sess_dir(mproj))
shutil.rmtree(mproj)

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()